# Initialize the instrument

In [ ]:
from Equipments import BNC575
from Equipments import Weeder
from Equipments import SR400
from Equipments import MCBOX
from Equipments import Andor
import numpy as np
import matplotlib.pyplot as plt
import pyvisa
import time
from scipy.signal import find_peaks

In [ ]:
from DataProcessing.Plot import PlotSaver

In [ ]:
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [ ]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

In [ ]:
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
sr400 = SR400.SR400('GPIB0::23::INSTR', timeout = 1000)

In [ ]:
weeder = Weeder.Weeder("ASRL1::INSTR")

In [ ]:
mcbox = MCBOX.MCBOX(find_device = "USB-3114")

# define the pulse

In [ ]:
pulse_arrangement = [["A", 900e-6, 0, "TTL"],              # MOT
                     [["B", 10e-6, 5e-6, 15],              # 780 EXC (20dB attenuator connected)
                      ["C", 10e-6, 5e-6, "TTL"]],          # 480 EXC
                     ["D", 20e-6, 0, 8],                   # DEI switch
                    ]
T = 1000e-6

cycle_number = 30
# photon counter gate width
gate_width = 15e-6
gate_delay = 3e-6

# stepper motor slope (for blue laser)
freq_factor = 129 * 2 #kHz

notes = {"gate_width": gate_width, "gate_delay": gate_delay}

In [ ]:
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 100e-6, count_period_num = cycle_number, 
                  dwell = 0, sourceA = "INPUT1", sourceB = "INPUT2", gate_A_mode = "FIXED",
                  gate_A_delay = gate_delay, gate_A_width = gate_width, gate_B_mode = "CW")

In [ ]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

In [ ]:
bnc575.start_pulses()

# search the resonance

In [ ]:
weeder.position(header = "A", query = True)

In [ ]:
weeder.move("A", position =3800, progress = True)

In [ ]:
weeder.advance(header = "A", num_step = -3)

# Stark map

## 1D

In [ ]:
center = 3440 - 100
stride = 4
half_width = 300


# scan list gneration
x_list = np.arange(2* half_width/stride)

In [ ]:
# everytime go further and go back to release the band tension and gear margin
start_position = center - half_width
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

results = []
s = time.time()
for idx in x_list:
    print(f"now checking step: {idx}", end = "\r", flush=True)
    sr400.count_reset()
    result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
    results.append(result)
    for i in range(stride):
        weeder.step("A","+")
    time.sleep(0.1)
e = time.time()
print(e-s)

In [ ]:
ps.Plot2D(x = x_list*stride*freq_factor, y = None, Z = results,
          xlabel = "480 laser detuning (kHz)", zlabel = "average number of counts", notes = notes)

In [ ]:
100000/(2*129)

## 2D

### frequency inside

In [ ]:
center = 3573
stride = 1
half_width = 200


# scan list gneration
x_list = np.arange(2* half_width/stride)

In [ ]:
start_position = center - half_width

v_list = np.linspace(-0.5,0.5,21) + vset[0]
stark_map = []

s = time.time()
for v in v_list:
    mcbox.set_voltage_1chan(board_num = 0, channel = 10, voltage = v, bipolar = True)
    mcbox.set_voltage_1chan(board_num = 0, channel = 11, voltage = -v, bipolar = True)
    
    # everytime go further and go back to release the band tension and gear margin
    weeder.move(header = "A", position = start_position-100)
    weeder.move(header = "A", position = start_position)
    results = []
    for idx in x_list:
        print(f"now checking step: {idx}", end = "\r", flush=True)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter="A",length = cycle_number))
        results.append(result)
        for i in range(stride):
            weeder.step("A","+")
        time.sleep(0.1)
    stark_map.append(results)

e = time.time()
print(e-s)

In [ ]:
ps.Plot3D(x = v_list, y =x_list*stride*freq_factor , Z = stark_map,
          xlabel = "x voltage (V)", ylabel = "frequency (kHz)", zlabel = "average number of counts",notes= notes)

### voltage inside

In [ ]:
center = 3440 - 100
stride = 1
half_width = 200


# scan list gneration
s_list = np.arange(2* half_width/stride)

# the 0 point

#vset = np.array([0.2,-0.1,0.48])
#vset = np.array([0.19,-0.12,0.67])
vset = np.array([0.2,-0.1,0.6])

vx_list = np.linspace(-1.0,1.0,11) + vset[0]
vy_list = np.linspace(-1.0,1.0,11) + vset[1]
vz_list = np.linspace(-2.0,2.0,11) + vset[2]

v_list = np.vstack((vx_list,vy_list,vz_list))


electrode_pair = [[10,11],[6,7],[0,1]]

In [ ]:
mcbox.set_voltage_1chan(board_num = 0, channel = 10, voltage = vset[0], bipolar = True, display=True)
mcbox.set_voltage_1chan(board_num = 0, channel = 11, voltage = -vset[0], bipolar = True, display=True)

mcbox.set_voltage_1chan(board_num = 0, channel = 6, voltage = vset[1], bipolar = True, display=True)
mcbox.set_voltage_1chan(board_num = 0, channel = 7, voltage = -vset[1], bipolar = True, display=True)

mcbox.set_voltage_1chan(board_num = 0, channel = 0, voltage = 0, bipolar = True, display=True)
mcbox.set_voltage_1chan(board_num = 0, channel = 1, voltage = -vset[2], bipolar = True, display=True)

In [ ]:
start_position = center - half_width

stark_map = []

i = 0

print('Now checking axis {}'.format(i))
weeder.move(header = "A", position = start_position-100)
weeder.move(header = "A", position = start_position)
for s in s_list:
    for j in range(stride):
            weeder.step("A","+")
    print(f"now checking step: {s}", end = "\r", flush = True)
    
    results = []
    for v_s in v_list[i]:
        # scan voltage
        if electrode_pair[i][0] == 0:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = 0, bipolar = True, display=False)
        else:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = v_s, bipolar = True, display=False)
        mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][1], voltage = -v_s, bipolar = True, display=False)
        time.sleep(0.01)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter = "A",length = cycle_number))
        results.append(result)
        
    stark_map.append(results)

In [ ]:
ps.Plot3D(x = v_list[i], y =s_list*stride*freq_factor , Z = np.array(stark_map).T,
          xlabel = f"Voltage {i} (V)", ylabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

### Full voltage scans (added 2025.12.11)

In [ ]:
center = 3440 - 100
stride = 2
half_width = 300

# scan list gneration
s_list = np.arange(2*half_width/stride)

# the 0 point

#vset = np.array([0.2,-0.1,0.48])
#vset = np.array([0.19,-0.12,0.67])
vset = np.array([0.2,-0.1,0.6])

vx_list = np.linspace(-10.0,10.0,21) + vset[0]
vy_list = np.linspace(-10.0,10.0,21) + vset[1]
vz_list = np.linspace(-10.0,10.0,21) + vset[2]

v_list = np.vstack((vx_list,vy_list,vz_list))

electrode_pair = [[10,11],[6,7],[0,1]]

In [ ]:
mcbox.set_voltage_1chan(board_num = 0, channel = 10, voltage = vset[0], bipolar = True, display=True)
mcbox.set_voltage_1chan(board_num = 0, channel = 11, voltage = -vset[0], bipolar = True, display=True)

mcbox.set_voltage_1chan(board_num = 0, channel = 6, voltage = vset[1], bipolar = True, display=True)
mcbox.set_voltage_1chan(board_num = 0, channel = 7, voltage = -vset[1], bipolar = True, display=True)

mcbox.set_voltage_1chan(board_num = 0, channel = 0, voltage = 0, bipolar = True, display=True)
mcbox.set_voltage_1chan(board_num = 0, channel = 1, voltage = -vset[2], bipolar = True, display=True)

In [ ]:
start_position = center - half_width

stark_map = []

# electrode pair to check
i = 0

print('Now checking axis {}'.format(i))
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

s = time.time()
print('start:',s)

for s in s_list:
    for j in range(stride):
            weeder.step("A","+")
    print(f"Now checking step: {s}", end = "\r", flush = True)
    
    results = []
    for v_s in v_list[i]:
        if v_s <= 10.0 and v_s >= -10.0:
            # scan voltage
            if electrode_pair[i][0] == 0:
                mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = 0, bipolar = True, display = False)
            else:
                mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = v_s, bipolar = True, display = False)
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][1], voltage = -v_s, bipolar = True, display = False)
            time.sleep(0.01)
            sr400.count_reset()
            result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
            results.append(result)
        else:
            continue
        
    stark_map.append(results)

e = time.time()
print('end:',e)
print('time elapsed:',e-s)

In [ ]:
#voltages = [v for v in v_list[i] if v <= 10.0 and v >= -10.0]
print(voltages)

In [ ]:
ps.Plot3D(x = voltages, y = s_list*stride*freq_factor , Z = np.array(stark_map).T,
          xlabel = f"Voltage {i} (V)", ylabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

In [ ]:
start = 1765564722.4786904
end = 1765566146.46307130
time_elapsed = end - start

print('{} seconds'.format(time_elapsed))

### Testing some code generating test voltages (12.12.25)

#### new code for looping (doesn't work)
voltages = []  # empty list to store voltages actually tested during scan
for s in s_list:
    index = s_list.index(s)
    for j in range(stride):
            weeder.step("A","+")
    print(f"Now checking step: {s}", end = "\r", flush = True)
    
    results = []
    for v_s in v_list[i]:
        # check if test voltage is within MC box range
        if v_s <= 10.0 and v_s >= -10.0:
            continue
        elif v_s > 10.0:
            v_s -= abs(vset[i])
        else:
            v_s += abs(vset[i])

        # store tested voltages (do this only once)
        if index == 0:
            voltages.append(v_s)
        else:
            continue
        
        # perform voltage scan
        if electrode_pair[i][0] == 0:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = 0, bipolar = True, display = False)
        else:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = v_s, bipolar = True, display = False)
        mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][1], voltage = -v_s, bipolar = True, display = False)
        time.sleep(0.01)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
        results.append(result)
        
    stark_map.append(results)

In [ ]:
import numpy as np
vset = np.array([0.2,-0.1,0.6])

vx_list = np.linspace(-10.0,10.0,21) + vset[0]
vy_list = np.linspace(-10.0,10.0,21) + vset[1]
vz_list = np.linspace(-10.0,10.0,21) + vset[2]

v_list = np.vstack((vx_list,vy_list,vz_list))

print(vset)
print(v_list)

In [ ]:
# helper function for checking that test voltages stay within MC box's range (-10 to 10 V) (VERSION 1)
# with this version, it's still possible for one endpoint (+/- 10 V) not be tested
def generate_test_voltages1(voltages, setpoints, axis: int):
    actual_voltages = []
    vmin, vmax = -10.0, 10.0
    
    for v in voltages[axis]:
        if v <= vmax and v >= vmin:
            actual_voltages.append(v)
        elif v > vmax:
            v -= abs(setpoints[axis])
            actual_voltages.append(v)
        else:
            v += abs(setpoints[axis])
            actual_voltages.append(v)

    return actual_voltages

In [ ]:
# helper function for checking that test voltages stay within MC box's range (-10 to 10 V) (VERSION 2)
# with this version, the entire voltage range is scanned, but the step size is still not completely uniform
def generate_test_voltages2(voltages, setpoints, axis: int):
    actual_voltages = []
    vmin, vmax = -10.0, 10.0
    
    for v in voltages[axis]:
        if v <= vmax and v >= vmin:
            actual_voltages.append(v)
        elif v > vmax:
            v -= abs(setpoints[axis])
            actual_voltages.append(v)
        else:
            v += abs(setpoints[axis])
            actual_voltages.append(v)

    v_actual_min = min(actual_voltages)
    v_actual_max = max(actual_voltages)

    min_index = actual_voltages.index(v_actual_min)
    max_index = actual_voltages.index(v_actual_max)

    actual_voltages[min_index] = vmin
    actual_voltages[max_index] = vmax
        
    return actual_voltages

In [ ]:
i = 1
print('original voltages: ', v_list[i])
print('helper function [option 1]: ', generate_test_voltages1(v_list, vset, axis = i))
print('helper function [option 2]: ', generate_test_voltages2(v_list, vset, axis = i))

In [ ]:
test_voltages = generate_test_voltages1(v_list, vset, axis = i)

print('Test voltages: \n {}'.format(test_voltages))

#### pseudocode for new looping code

start_position = center - half_width

stark_map = []

#### electrode pair to check
i = 0

print('Now checking axis {}'.format(i))
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

s = time.time()
print('start:',s)

voltages = generate_test_voltages(v_list, vset, axis = i)

for s in s_list:
    for j in range(stride):
            weeder.step("A","+")
    print(f"Now checking step: {s}", end = "\r", flush = True)
    
    results = []
    for v_s in voltages:
        # scan voltage
        if electrode_pair[i][0] == 0:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = 0, bipolar = True, display = False)
        else:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = v_s, bipolar = True, display = False)
        mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][1], voltage = -v_s, bipolar = True, display = False)
        time.sleep(0.01)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
        results.append(result)
        
    stark_map.append(results)

e = time.time()
print('end:',e)
print('time elapsed:',e-s)

### Peak finding and fitting

In [ ]:
print(v_list)

In [ ]:
peaks_x = []
peaks_y = []
peaks_list = []
for k in range(len(stark_map[0])):
    freq_scan = np.array(stark_map)[:,k]
    peaks,_ = find_peaks(freq_scan, height = 3, distance = 16)
    print(peaks)
    if len(peaks) == 3:
        for p in peaks:
            peaks_x.append(v_list[i][k])
            peaks_y.append((s_list*stride*freq_factor)[p])
        peaks_list.append(peaks)
peaks_list = np.array(peaks_list)
peaks_x = np.reshape(peaks_x,(-1,3)).T
peaks_y = np.reshape(peaks_y,(-1,3)).T

In [ ]:
print(v_list)

In [ ]:
plt.pcolormesh(v_list[i],s_list*stride*freq_factor,stark_map)
ssum = 0 
for l in range(3):
    a, b, c = np.polyfit(peaks_x[l], peaks_y[l], 2)
    plt.scatter(peaks_x[l],peaks_y[l])
    plt.plot(v_list[i], a * v_list[i]**2 + b * v_list[i] +c)
    plt.scatter(-b/(2*a), a * -b/(2*a)**2 + b * -b/(2*a) +c, c = "r")
    plt.xlabel(f"Voltage {i} (V)")
    plt.ylabel("Frequency (kHz)")
    ssum += -b/(2*a)
print(ssum/3)
plt.colorbar()

# End

In [ ]:
mcbox.set_voltage_Nchan(board_num = 0, channels = [0,1,6,7,10,11], voltages = [0,0,0,0,0,0], bipolar = True)

In [ ]:
weeder.save(header = "A", query = True)

In [ ]:
bnc575.disable_all()

In [ ]:
bnc575.disarm_all()

In [ ]:
sr400.count_reset()